<a href="https://colab.research.google.com/github/AnjanPayra/MM-CCNB/blob/main/MM_CCNB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import math
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

UPLOAD_DIR = "/content"
OUT_DIR = "/mnt/user-data/outputs"

NETWORK_FILES = {
    "YDIP": f"{UPLOAD_DIR}/YDIP.txt",
    "YHQ": f"{UPLOAD_DIR}/YHQ.txt",
    "YMBD": f"{UPLOAD_DIR}/YMBD.txt",
    "YMIPS": f"{UPLOAD_DIR}/YMIPS.txt",
}
ESSENTIAL_XLSX = f"{UPLOAD_DIR}/Essential.xlsx"

TOP_PERCENTS = [1, 5, 10, 15, 20, 25, 30, 35, 40]
K_VALUES = [1, 2, 3]

In [2]:
# ---------------------------------------------------------------------------
# 1. Load network
# ---------------------------------------------------------------------------
def load_network(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]          # 'Protein 1' -> 'Protein 1' (still has inner space)
    cols = list(df.columns)
    df = df.rename(columns={cols[0]: "Protein1", cols[1]: "Protein2"})
    df["Protein1"] = df["Protein1"].astype(str).str.strip()
    df["Protein2"] = df["Protein2"].astype(str).str.strip()
    df = df.dropna(subset=["Protein1", "Protein2"])

    G = nx.Graph()
    G.add_edges_from(zip(df["Protein1"], df["Protein2"]))
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


In [3]:
# ---------------------------------------------------------------------------
# 2. Edge-Clustering Coefficient (ECC)
#    ECC(u,v) = Z_uv / max(d_u, d_v)   where Z_uv = # triangles on edge (u,v)
#              = |N(u) ∩ N(v)|
#    ECC(u)   = sum over neighbours t of u of ECC(u,t)
# ---------------------------------------------------------------------------
def compute_ecc(G):
    degree = dict(G.degree())
    ecc_uv = {}
    for u, v in G.edges():
        z_uv = len(list(nx.common_neighbors(G, u, v)))
        denom = max(degree[u], degree[v])
        ecc_uv[(u, v)] = z_uv / denom if denom > 0 else 0.0

    ecc_u = {n: 0.0 for n in G.nodes()}
    for (u, v), val in ecc_uv.items():
        ecc_u[u] += val
        ecc_u[v] += val
    return ecc_uv, ecc_u



In [4]:
# ---------------------------------------------------------------------------
# GO_Nb / SL_Nb stubs -- fill these in if/when a GO or subcellular-
# localization annotation file becomes available (protein -> set of terms).
# ---------------------------------------------------------------------------
def compute_go_nb(G, go_annotations=None):
    """go_annotations: dict protein -> set(GO terms). Returns GO_Nb(u,v), GO_Nb(u)."""
    if not go_annotations:
        return {}, {n: 0.0 for n in G.nodes()}
    go_nb_uv = {}
    for u, v in G.edges():
        gu, gv = go_annotations.get(u, set()), go_annotations.get(v, set())
        if not gu or not gv:
            go_nb_uv[(u, v)] = 0.0
            continue
        common = set(G[u]) & set(G[v])
        p = sum(len((go_annotations.get(u, set()) & go_annotations.get(t, set())) |
                    (go_annotations.get(v, set()) & go_annotations.get(t, set())))
                for t in common)
        q = min(len(gu), len(gv))
        go_nb_uv[(u, v)] = p / q if q > 0 else 0.0
    go_nb_u = {n: 0.0 for n in G.nodes()}
    for (u, v), val in go_nb_uv.items():
        go_nb_u[u] += val
        go_nb_u[v] += val
    return go_nb_uv, go_nb_u


def compute_sl_nb(G, sl_annotations=None):
    """sl_annotations: dict protein -> set(subcellular compartments)."""
    if not sl_annotations:
        return {}, {n: 0.0 for n in G.nodes()}
    sl_nb_uv = {}
    for u, v in G.edges():
        su, sv = sl_annotations.get(u, set()), sl_annotations.get(v, set())
        if not su or not sv:
            sl_nb_uv[(u, v)] = 0.0
            continue
        common = set(G[u]) & set(G[v])
        r = len({t for t in common if sl_annotations.get(t, set())}) ** 2
        s = len(su) * len(sv)
        sl_nb_uv[(u, v)] = r / s if s > 0 else 0.0
    sl_nb_u = {n: 0.0 for n in G.nodes()}
    for (u, v), val in sl_nb_uv.items():
        sl_nb_u[u] += val
        sl_nb_u[v] += val
    return sl_nb_uv, sl_nb_u


def combine_scores(ecc_u, go_nb_u, sl_nb_u):
    """ES(u) = MIN(ECC(u), MAX(GO_Nb(u), SL_Nb(u))) when GO/SL are available,
    otherwise falls back to ECC(u) alone (pure topological score)."""
    have_go_sl = any(v > 0 for v in go_nb_u.values()) or any(v > 0 for v in sl_nb_u.values())
    es = {}
    for n in ecc_u:
        if have_go_sl:
            es[n] = min(ecc_u[n], max(go_nb_u.get(n, 0.0), sl_nb_u.get(n, 0.0)))
        else:
            es[n] = ecc_u[n]
    return es, have_go_sl



In [5]:
# ---------------------------------------------------------------------------
# 3. 3-point threshold:  Thresh_k(p) = mean + K*std*(1 - 1/(1+std^2))
# ---------------------------------------------------------------------------
def three_point_threshold(scores, k):
    arr = np.array(list(scores.values()), dtype=float)
    mean, std = arr.mean(), arr.std()
    return mean + k * std * (1 - 1 / (1 + std ** 2))

In [6]:

# ---------------------------------------------------------------------------
# 4. Load gold-standard essential / non-essential lists
# ---------------------------------------------------------------------------
def load_gold_standard(path):
    xls = pd.ExcelFile(path)
    ess_sheet = [s for s in xls.sheet_names if "non" not in s.lower() and "essential" in s.lower()][0]
    non_sheet = [s for s in xls.sheet_names if "non" in s.lower() and "essential" in s.lower()][0]
    essential = set(pd.read_excel(path, sheet_name=ess_sheet, header=None)[0].dropna().astype(str).str.strip())
    nonessential = set(pd.read_excel(path, sheet_name=non_sheet, header=None)[0].dropna().astype(str).str.strip())
    return essential, nonessential

In [7]:

# ---------------------------------------------------------------------------
# 5. Confusion-matrix metrics
# ---------------------------------------------------------------------------
def confusion_metrics(predicted_set, essential_set, nonessential_set, universe):
    labeled = universe & (essential_set | nonessential_set)
    TP = sum(1 for n in labeled if n in predicted_set and n in essential_set)
    FP = sum(1 for n in labeled if n in predicted_set and n in nonessential_set)
    FN = sum(1 for n in labeled if n not in predicted_set and n in essential_set)
    TN = sum(1 for n in labeled if n not in predicted_set and n in nonessential_set)

    def safe_div(a, b):
        return a / b if b else float("nan")

    sensitivity = safe_div(TP, TP + FN)
    specificity = safe_div(TN, TN + FP)
    ppv = safe_div(TP, TP + FP)
    npv = safe_div(TN, TN + FN)
    f1 = safe_div(2 * ppv * sensitivity, ppv + sensitivity) if not (math.isnan(ppv) or math.isnan(sensitivity)) else float("nan")
    accuracy = safe_div(TP + TN, TP + TN + FP + FN)
    mcc_denom = math.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)) if (TP + FP) and (TP + FN) and (TN + FP) and (TN + FN) else 0
    mcc = safe_div((TP * TN - FP * FN), mcc_denom) if mcc_denom else float("nan")

    return dict(TP=TP, FP=FP, FN=FN, TN=TN, Sensitivity=sensitivity, Specificity=specificity,
                PPV=ppv, NPV=npv, F1=f1, Accuracy=accuracy, MCC=mcc, n_labeled=len(labeled))

In [8]:
# ---------------------------------------------------------------------------
# 6. Top-N% recall table
# ---------------------------------------------------------------------------
def top_n_recall(scores, essential_set, universe, percents):
    labeled_essential_in_net = universe & essential_set
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    ranked_nodes = [n for n, _ in ranked]
    rows = []
    for p in percents:
        k = max(1, round(len(ranked_nodes) * p / 100))
        top_k = set(ranked_nodes[:k])
        tp = len(top_k & labeled_essential_in_net)
        rows.append(dict(
            TopPercent=p, NumSelected=k, TrueEssentialFound=tp,
            TotalKnownEssentialInNet=len(labeled_essential_in_net),
            Recall=tp / len(labeled_essential_in_net) if labeled_essential_in_net else float("nan"),
            Precision=tp / k if k else float("nan"),
        ))
    return pd.DataFrame(rows)


In [13]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    import os
    essential_set, nonessential_set = load_gold_standard(ESSENTIAL_XLSX)
    print(f"Gold standard: {len(essential_set)} essential, {len(nonessential_set)} non-essential proteins\n")

    os.makedirs(OUT_DIR, exist_ok=True)
    writer = pd.ExcelWriter(f"{OUT_DIR}/essential_protein_results.xlsx", engine="openpyxl")
    summary_rows = []
    topn_frames = {}

    for name, path in NETWORK_FILES.items():
        print(f"=== {name} ===")
        G = load_network(path)
        universe = set(G.nodes())
        print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

        ecc_uv, ecc_u = compute_ecc(G)
        go_nb_uv, go_nb_u = compute_go_nb(G)   # no GO file supplied -> all zeros
        sl_nb_uv, sl_nb_u = compute_sl_nb(G)   # no SL file supplied -> all zeros
        es, have_go_sl = combine_scores(ecc_u, go_nb_u, sl_nb_u)
        score_label = "ES (MIN(ECC,MAX(GO_Nb,SL_Nb)))" if have_go_sl else "ES (= ECC(u), GO/SL data not supplied)"
        print(f"Essentiality score used: {score_label}")

        # ---- per-protein score table ----
        deg = dict(G.degree())
        score_df = pd.DataFrame({
            "Protein": list(G.nodes()),
            "Degree": [deg[n] for n in G.nodes()],
            "ECC(u)": [ecc_u[n] for n in G.nodes()],
            "ES(u)": [es[n] for n in G.nodes()],
            "GoldLabel": [
                "Essential" if n in essential_set else ("NonEssential" if n in nonessential_set else "Unlabeled")
                for n in G.nodes()
            ],
        }).sort_values("ES(u)", ascending=False)

        # ---- K=1,2,3 threshold predictions + metrics ----
        for k in K_VALUES:
            thr = three_point_threshold(es, k)
            predicted = {n for n, v in es.items() if v > thr}
            metrics = confusion_metrics(predicted, essential_set, nonessential_set, universe)
            score_df[f"Predicted_K{k}"] = score_df["Protein"].isin(predicted)
            summary_rows.append(dict(Network=name, K=k, Threshold=round(thr, 4),
                                      NumPredicted=len(predicted), **metrics))
            print(f"  K={k}: thr={thr:.4f}  predicted={len(predicted):5d}  "
                  f"Sens={metrics['Sensitivity']:.3f}  Spec={metrics['Specificity']:.3f}  "
                  f"PPV={metrics['PPV']:.3f}  Acc={metrics['Accuracy']:.3f}  MCC={metrics['MCC']:.3f}")

        # ---- Top-N% recall ----
        topn_df = top_n_recall(es, essential_set, universe, TOP_PERCENTS)
        topn_frames[name] = topn_df

        score_df.to_excel(writer, sheet_name=f"{name}_scores", index=False)
        topn_df.to_excel(writer, sheet_name=f"{name}_topN", index=False)
        print()

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_excel(writer, sheet_name="Summary_metrics", index=False)
    writer.close()

    # ---- Comparison plot: Top-N% recall curves, all 4 networks ----
    plt.figure(figsize=(7, 5))
    for name, df in topn_frames.items():
        plt.plot(df["TopPercent"], df["Recall"] * 100, marker="o", label=name)
    plt.xlabel("Top N% of ranked proteins (by ES(u))")
    plt.ylabel("% of known essential proteins recovered")
    plt.title("Essential-protein recall vs. top-ranked percentage")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/topN_recall_comparison.png", dpi=150)
    plt.close()

    print("\n=== SUMMARY (K-threshold metrics) ===")
    print(summary_df.to_string(index=False))

    return summary_df


if __name__ == "__main__":
    main()

Gold standard: 1285 essential, 4394 non-essential proteins

=== YDIP ===
Nodes: 5093, Edges: 24743
Essentiality score used: ES (= ECC(u), GO/SL data not supplied)
  K=1: thr=1.2984  predicted=  630  Sens=0.276  Spec=0.916  PPV=0.517  Acc=0.759  MCC=0.245
  K=2: thr=2.0650  predicted=  411  Sens=0.201  Spec=0.952  PPV=0.579  Acc=0.768  MCC=0.237
  K=3: thr=2.8317  predicted=  260  Sens=0.139  Spec=0.974  PPV=0.635  Acc=0.769  MCC=0.216

=== YHQ ===
Nodes: 4743, Edges: 22665
Essentiality score used: ES (= ECC(u), GO/SL data not supplied)
  K=1: thr=16.9397  predicted=  344  Sens=0.154  Spec=0.952  PPV=0.525  Acc=0.750  MCC=0.177
  K=2: thr=29.8202  predicted=  240  Sens=0.088  Spec=0.961  PPV=0.436  Acc=0.739  MCC=0.097
  K=3: thr=42.7008  predicted=  148  Sens=0.029  Spec=0.969  PPV=0.241  Acc=0.730  MCC=-0.005

=== YMBD ===
Nodes: 2559, Edges: 11206
Essentiality score used: ES (= ECC(u), GO/SL data not supplied)
  K=1: thr=22.0275  predicted=  206  Sens=0.067  Spec=0.917  PPV=0.268  Ac